<a href="https://colab.research.google.com/github/VishnuCodes96/Rebounce-Applied-AI-and-Analytics-Codes/blob/main/MP4_AirQuality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Importing the required libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as pe

## Loading the dataset using Pandas

In [2]:
url = "https://raw.githubusercontent.com/urbanemissions-info/AQI_bulletins/main/data/Processed/AllIndiaBulletins_Master.csv"
df = pd.read_csv(url, low_memory=False)

### Displaying First and Last 5 rows

In [3]:
df.head()

,date,City,No. Stations,Air Quality,Index Value,Prominent Pollutant
0,2015-05-01,Varanasi,NaN,Moderate,157,PM10
1,2015-05-01,Kanpur,NaN,Moderate,175,PM10
2,2015-05-01,Faridabad,NaN,Moderate,173,PM10
3,2015-05-01,Ahmedabad,NaN,Moderate,168,PM2.5
4,2015-05-01,Hyderabad,NaN,Moderate,189,PM2.5


In [4]:
df.tail()

,date,City,No. Stations,Air Quality,Index Value,Prominent Pollutant
299969,2023-12-31,Moradabad,6,Satisfactory,97,"PM10, PM2.5"
299970,2023-12-31,Milupara,1,Moderate,134,PM2.5
299971,2023-12-31,Meerut,3,Moderate,173,PM2.5
299972,2023-12-31,Pratapgarh,1,Poor,248,PM2.5
299973,2023-12-31,Sangli,1,Moderate,111,PM2.5


### Infromation about the whole dataset

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299974 entries, 0 to 299973
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   date                 299974 non-null  object
 1   City                 299974 non-null  object
 2   No. Stations         299834 non-null  object
 3   Air Quality          299974 non-null  object
 4   Index Value          299974 non-null  int64 
 5   Prominent Pollutant  299973 non-null  object
dtypes: int64(1), object(5)
memory usage: 13.7+ MB


### Missing values in No. Stations is `140` and Prominent Pollutant column is `1`

In [6]:
print("No. of Rows & Columns: ",df.shape)
print("Name of the Columns: ",df.columns)

No. of Rows & Columns:  (299974, 6)
Name of the Columns:  Index(['date', 'City', 'No. Stations', 'Air Quality', 'Index Value',
       'Prominent Pollutant'],
      dtype='object')


### Counting rows that repeat the same date and City

In [7]:
df.duplicated(subset=["date", "City"]).sum()

np.int64(34)

In [8]:
df.duplicated(subset=["date", "City"]).value_counts()

,count
False,299940
True,34


### How many different cities are there? What are the first and last dates?

In [9]:
print(f'Number of Cities in the dataset: {df['City'].nunique()}')

Number of Cities in the dataset: 277


In [10]:
print(f'First date recorded in the dataset: {df['date'].min()}')

First date recorded in the dataset: 2015-05-01


In [11]:
print(f'Latest date recorded in the dataset: {df['date'].max()}')

Latest date recorded in the dataset: 2023-12-31


### Show the count of each value in Air Quality. What do you notice about the spellings?

In [12]:
df['Air Quality'].value_counts()

,count
Air Quality,
Satisfactory,105251
Moderate,96139
Good,47440
Poor,34641
Very Poor,13969
Severe,2516
Very poor,18


#### Every category is in Title Case, except 'Very Poor' cateogry is repeated twice once in Title Case and once in normal case

## Cleaning the Dataset

### Converting date into a proper datetime column.

In [13]:
df['date'] = pd.to_datetime(df['date'])
print(f'Date column datatype changed to {df['date'].dtype}')

Date column datatype changed to datetime64[ns]


In [14]:
df.shape

(299974, 6)

### Fixing the Air Quality spellings so each category appears once.

In [15]:
df['Air Quality'] = df['Air Quality'].str.title()

In [16]:
df['Air Quality'].value_counts()

,count
Air Quality,
Satisfactory,105251
Moderate,96139
Good,47440
Poor,34641
Very Poor,13987
Severe,2516


In [17]:
df.shape

(299974, 6)

### Removing repeated city-days, keeping the first one

In [18]:
df = df.drop_duplicates(subset=['date','City'], keep='first')

#### No. of duplicate rows = 34, No. of rows in unaltered dataset = 299974, No. of rows after removing duplicates based only on the combination of date and city = 299974 - 34 => 299940

In [19]:
df.shape

(299940, 6)

### Dropping the `No. Stations` column.

In [20]:
df = df.drop(columns=['No. Stations'])

In [21]:
df.shape

(299940, 5)

#### Reason for dropping `No. Stations` column: This project focuses on analyzing city-level AQI trends, air quality categories, and prominent pollutants, and the number of monitoring stations is not essential to the primary analysis. Although dropping it is reasonable for the current project, the number of monitoring stations could influence how representative a city's AQI measurement is. So it may be useful in a more detailed study.

### Creating two new columns: year and month

In [22]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
# Pandas provides the '.dt' accessor to extract date-related information.

In [23]:
df[['date','year','month']].head()

,date,year,month
0,2015-05-01,2015,5
1,2015-05-01,2015,5
2,2015-05-01,2015,5
3,2015-05-01,2015,5
4,2015-05-01,2015,5


In [24]:
df.shape

(299940, 7)

### replacing "OZONE" with "O3" in Prominent Pollutant.

In [25]:
df['Prominent Pollutant'] = df['Prominent Pollutant'].replace('OZONE','O3')

In [26]:
df[df['Prominent Pollutant'] == 'O3'].head(2)

,date,City,Air Quality,Index Value,Prominent Pollutant,year,month
6,2015-05-01,Delhi,Poor,221,O3,2015,5
28,2015-05-03,Agra,Satisfactory,84,O3,2015,5
